# 03. KPIs de transparencia

## Presentación de la sección
Este notebook concentra los indicadores de síntesis utilizados en la sustentación del proyecto. La finalidad es disponer de una vista ejecutiva de apoyo para la exposición de resultados.


In [16]:
import importlib.util
requeridas = ['pandas', 'numpy']
faltantes = [pkg for pkg in requeridas if importlib.util.find_spec(pkg) is None]
if faltantes:
    raise ModuleNotFoundError(f'Faltan librerias en este kernel: {faltantes}. Instale requirements.txt o seleccione otro interprete.')
print('Entorno verificado correctamente.')


Entorno verificado correctamente.


## Librerias de trabajo


In [17]:
from pathlib import Path
import unicodedata
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'data').exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / 'data').exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError('No se encontro la carpeta data del proyecto.')

MASTER_FILE = PROJECT_DIR / 'data' / 'contrataciones_peru_2022_2024_maestro.csv'
df = pd.read_csv(MASTER_FILE, low_memory=False)
def obtener_columna_anio(dataframe):
    for columna in dataframe.columns:
        normal = unicodedata.normalize('NFKD', str(columna)).encode('ascii', 'ignore').decode('ascii').lower()
        if normal in ('anio', 'ano'):
            return columna
    raise KeyError('No se encontro una columna de a?o en la base.')

col_anio = obtener_columna_anio(df)
df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['n_postores'] = pd.to_numeric(df['n_postores'], errors='coerce').fillna(0)
df['un_solo_postor'] = df['n_postores'].eq(1)
df['directa'] = df['metodo_simple'].astype(str).str.contains('direct', case=False, na=False)


## KPI 1. Total de procesos analizados


In [18]:
kpi_1 = df['ocid'].nunique()
print('Total de procesos analizados:', format(kpi_1, ','))


Total de procesos analizados: 224,596


## KPI 2. Porcentaje de procesos con un solo postor


In [19]:
kpi_2 = df['un_solo_postor'].mean() * 100
print('Porcentaje de procesos con un solo postor:', round(kpi_2, 2), '%')


Porcentaje de procesos con un solo postor: 13.39 %


## KPI 3. Participación de contrataciones directas


In [20]:
kpi_3 = df['directa'].mean() * 100
print('Participación de contrataciones directas:', round(kpi_3, 2), '%')


Participación de contrataciones directas: 8.14 %


## KPI 4. Monto adjudicado total


In [21]:
kpi_4 = df['monto_adjudicado'].sum()
print('Monto adjudicado total (PEN):', format(round(kpi_4, 2), ',.2f'))


Monto adjudicado total (PEN): 122,712,292,758.96


## KPI 5. Score promedio de transparencia territorial

Se resume el desempeño territorial a partir de tres componentes: contratación directa, un solo postor y proporción de monto en riesgo.


In [22]:
resumen = (df.groupby('departamento', as_index=False).agg(pct_directa=('directa', 'mean'), pct_un_solo_postor=('un_solo_postor', 'mean'), monto_total=('monto_adjudicado', 'sum'), monto_riesgo=('monto_adjudicado', lambda s: s[df.loc[s.index, 'un_solo_postor']].sum())))
resumen['pct_monto_riesgo'] = np.where(resumen['monto_total'] > 0, resumen['monto_riesgo'] / resumen['monto_total'], 0)
resumen['score_transparencia'] = (0.4 * resumen['pct_directa'] + 0.3 * resumen['pct_un_solo_postor'] + 0.3 * resumen['pct_monto_riesgo']) * 100
print('Score promedio de transparencia territorial:', round(resumen['score_transparencia'].mean(), 2))
resumen[['departamento', 'score_transparencia']].sort_values('score_transparencia', ascending=False).head(10)


Score promedio de transparencia territorial: 7.79


,departamento,score_transparencia
23,TUMBES,20.102634
14,LIMA,19.280786
15,LORETO,14.270718
6,CALLAO,10.077035
21,SAN MARTIN,8.650456
10,ICA,8.513632
12,LA LIBERTAD,8.487846
13,LAMBAYEQUE,8.316416
0,AMAZONAS,8.292633
19,PIURA,7.914542
